# Notebook 3 — CVaR-Constrained PPO: Four-Agent Comparison

Core thesis result. Compares all four agents on DOGEUSDT market replay:

| Agent | Reward | Constraint | Role |
|---|---|---|---|
| A-S baseline | — (analytical) | Q_MAX only | Floor |
| Unconstrained PPO | CjMmCriterion | none | Falces Marin replication |
| DD-penalised PPO | CjMmCriterion − α·DD | none | Reward shaping approach |
| CVaR-constrained PPO | CjMmCriterion | CVaR(DD) ≤ d | **Novel contribution** |

In [ ]:
import sys, pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stable_baselines3 import PPO
from procs.stochastic_processes import (
    MarketReplayMidpriceModel, PoissonArrivalModel, ExponentialFillFunction,
)
from procs.gym.model_dynamics import LimitOrderModelDynamics
from procs.gym.trading_environment import TradingEnvironment
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.gym.cvar_lagrangian import (
    train_cvar_ppo, calibrate_cvar_threshold, calibrate_cvar_threshold_windowed,
)
from procs.rewards import PnLReward, CjMmCriterion, CjMmDrawdownPenalty
from procs.agents import AvellanedaStoikovAgent, Sb3Agent
from procs.gym.helpers.plotting import (
    plot_trajectory, plot_learned_policy, plot_cvar_training,
)
from procs.gym.helpers.generate_trajectory_stats import generate_trajectory_stats
from procs.gym.helpers.fast_rollout import fast_simulate
from procs.gym.data_loader import load_single_day
from procs.gym.calibration import tune_gamma
from procs.gym.features import FeatureComputer, RollingVolatility
from procs.gym.reward_scale import estimate_reward_scale

%matplotlib inline

## 1. Load Data & Parameters

In [ ]:
DATA_PATH = r"C:\Users\john-\Documents\test_thesis\datasets\binance_book_snapshot_25_2025-01-01_DOGEUSDT.csv"
S, dt_sec, dt_index = load_single_day(DATA_PATH)
T_sec = float(dt_sec.sum())
sigma = MarketReplayMidpriceModel(S, dt_sec).volatility

kappa, A, tick, Q_MAX, phi = 35_000, 0.8, 0.00001, 50, 0.01
alpha_dd = 1.0   # DD penalty weight for agent 3

print(f"Loaded {len(S):,} snapshots, σ={sigma:.6f}")

## 2. Calibrate γ

In [ ]:
best_gamma, study = tune_gamma(
    midprices=S, dt_array=dt_sec, sigma=sigma, kappa=kappa, A=A,
    tick_size=tick, Q_MAX=Q_MAX,
    gamma_range=(0.001, 1.0), n_trials=100, num_trajectories=50,
)
print(f"Using γ = {best_gamma:.6f}")

reward_scale = estimate_reward_scale(
    midprices=S, dt_array=dt_sec, sigma=sigma,
    kappa=kappa, A=A, terminal_time=T_sec,
    tick_size=tick, Q_MAX=Q_MAX,
    num_trajectories=50, use_bm=False,
)

## 3. Environment Factory

In [ ]:
def get_env(normalise=False, reward_fn=None):
    if reward_fn is None:
        reward_fn = PnLReward()
    return TradingEnvironment(
        model_dynamics=LimitOrderModelDynamics(
            midprice_model=MarketReplayMidpriceModel(S, dt_sec, 1),
            arrival_model=PoissonArrivalModel(
                np.array([A, A]), 1, use_linear_approximation=False,
            ),
            fill_probability_model=ExponentialFillFunction(kappa, 1),
        ),
        reward_function=reward_fn, max_inventory=Q_MAX,
        normalise_observation_space=normalise,
        normalise_action_space=normalise,
        normalise_rewards=normalise,
        reward_scale=reward_scale if normalise else None,
        feature_computer=FeatureComputer([RollingVolatility(100)]) if normalise else None,
    )

## 4. Agent 1 — A-S Baseline

In [ ]:
as_agent = AvellanedaStoikovAgent(best_gamma, sigma, kappa, T_sec, tick)
as_stats = fast_simulate(
    midprices=S, dt_array=dt_sec, gamma=best_gamma, sigma=sigma,
    kappa=kappa, A=A, terminal_time=T_sec, tick_size=tick, Q_MAX=Q_MAX,
    num_trajectories=50, seed=42, use_linear_approximation=False,
)
print(f"A-S:  Sharpe={as_stats['sharpe'].mean():.4f}  MaxDD={as_stats['max_drawdown'].mean():.4f}")

## 5. Agent 2 — Unconstrained PPO

In [ ]:
%%time
train_uc = get_env(normalise=True, reward_fn=CjMmCriterion(phi))
sb3_uc = StableBaselinesTradingEnvironment(train_uc)

model_uc = PPO(
    "MlpPolicy", sb3_uc, verbose=1, device="cpu",
    n_steps=2048, batch_size=512, n_epochs=10,
    learning_rate=3e-4, gamma=0.999, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.01,
)
model_uc.learn(total_timesteps=2048 * 50)
print("Unconstrained PPO trained.")

## 6. Agent 3 — DD-Penalised PPO

In [ ]:
%%time
train_dd = get_env(normalise=True, reward_fn=CjMmDrawdownPenalty(phi, alpha_dd))
sb3_dd = StableBaselinesTradingEnvironment(train_dd)

model_dd = PPO(
    "MlpPolicy", sb3_dd, verbose=1, device="cpu",
    n_steps=2048, batch_size=512, n_epochs=10,
    learning_rate=3e-4, gamma=0.999, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.01,
)
model_dd.learn(total_timesteps=2048 * 50)
print("DD-penalised PPO trained.")

## 7. Agent 4 — CVaR-Constrained PPO

Set threshold from unconstrained PPO's drawdown distribution.
Warm-start from unconstrained PPO weights.

In [ ]:
# Calibrate threshold from unconstrained PPO
ppo_uc_agent = Sb3Agent(model_uc, train_env=train_uc)
dd_threshold = calibrate_cvar_threshold_windowed(
    agent=ppo_uc_agent,
    env_fn=lambda: get_env(normalise=False),
    n_episodes=10, n_steps=2048,
    cvar_alpha=0.2, tighten=0.20,
)
print(f"CVaR threshold: {dd_threshold[0]:.6f}")

In [ ]:
%%time
train_cvar = get_env(normalise=True, reward_fn=CjMmCriterion(phi))
sb3_cvar = StableBaselinesTradingEnvironment(train_cvar)

model_cvar, cb_cvar, _ = train_cvar_ppo(
    sb3_env=sb3_cvar,
    total_timesteps=2048 * 50,
    cvar_alpha=0.2,
    dd_threshold=dd_threshold[0],
    lr_lambda=0.01,
    lambda_max=500.0,
    model_init=model_uc,   # warm-start from unconstrained PPO
    ppo_kwargs=dict(
        n_steps=2048, batch_size=512, n_epochs=10,
        learning_rate=3e-4, gamma=0.999, gae_lambda=0.95,
        clip_range=0.2, ent_coef=0.01,
    ),
    verbose=0,
)
print(f"CVaR PPO trained. Final λ = {cb_cvar.lambda_:.4f}")

In [ ]:
plot_cvar_training(cb_cvar.cvar_history, cb_cvar.lambda_history, dd_threshold[0])

## 8. Four-Agent Comparison

In [ ]:
agents = {
    "A-S Baseline": as_agent,
    "Unconstrained PPO": Sb3Agent(model_uc, train_env=train_uc),
    "DD-Penalised PPO": Sb3Agent(model_dd, train_env=train_dd),
    "CVaR PPO": Sb3Agent(model_cvar, train_env=train_cvar),
}

rows = {}
for name, ag in agents.items():
    if name == "A-S Baseline":
        # Use fast_simulate (no features needed)
        st = fast_simulate(
            midprices=S, dt_array=dt_sec, gamma=best_gamma, sigma=sigma,
            kappa=kappa, A=A, terminal_time=T_sec, tick_size=tick, Q_MAX=Q_MAX,
            num_trajectories=1, seed=42, use_linear_approximation=False,
        )
    else:
        ev = get_env(normalise=False)
        st = generate_trajectory_stats(ev, ag, seed=42)
    rows[name] = {
        "Sharpe": st["sharpe"][0],
        "Sortino": st["sortino"][0],
        "Max DD": st["max_drawdown"][0],
        "P&L-to-MAP": st["pnl_to_map"][0],
        "Final PnL": st["total_pnl"][0],
    }

comp = pd.DataFrame(rows).T
print("\nFour-Agent Comparison — DOGEUSDT Day 1:")
print(comp.to_string(float_format="%.6f"))

## 9. Trajectory Plots

In [ ]:
for name, ag in agents.items():
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    ev = get_env(normalise=False)
    plot_trajectory(ev, ag, seed=42, datetime_index=dt_index)

In [ ]:
model_uc.save("models/ppo_unconstrained_doge")
model_dd.save("models/ppo_dd_penalised_doge")
model_cvar.save("models/ppo_cvar_doge")
print("All models saved.")